# 06 - Interpretation: residues and anchors

Why does the model predict binding? We use two complementary tools:

* **SHAP per-field blocks** (`explain_baseline`) - how much each sequence source
  (CDR3b / peptide / MHC pseudo-seq) contributes overall.
* **Model-agnostic occlusion** (`residue_attribution`) - per-residue importance for
  one record, then mapped onto the canonical MHC-I anchors **P2** and **PΩ**
  (the C-terminus).

In [ ]:
import numpy as np
from tcr_cliff.data import load_toy, split_pairs
from tcr_cliff.config import Config, DataConfig, EmbeddingConfig, ModelConfig
from tcr_cliff.models import train_model
from tcr_cliff.interpret import (
    explain_baseline,
    residue_attribution,
    anchor_importance,
    map_to_positions,
)

df = load_toy()
parts = split_pairs(df, DataConfig(group_split_on='peptide', split_column='__none__'), seed=0)
train_df, test_df = parts['train'], parts['test']

cfg = Config(
    seed=0,
    embedding=EmbeddingConfig(backend='fallback', fallback_dim=64, cache_dir=None),
    model=ModelConfig(kind='baseline_lgbm'),
)
cfg.model.baseline.n_estimators = 50
model = train_model(cfg, train_df)

## SHAP per-field importance

`explain_baseline` aggregates SHAP values over the embedding columns back into
per-field blocks (summing `|shap|` within each field's `"<field>_<j>"` columns).

In [ ]:
expl = explain_baseline(model, train_df.head(30), max_display=20)
field_scores = expl.get('field_importance', expl)
print('per-field importance blocks:')
for field in ('cdr3b', 'peptide', 'mhc_pseudo'):
    if field in field_scores:
        print(f'  {field:11s}: {float(field_scores[field]):.4f}')

## Per-residue occlusion on a peptide

`residue_attribution` is model-agnostic: it substitutes each residue (alanine /
mask) and measures the drop in predicted score, so it works even without torch.
We wrap the model's `predict_proba` as the `predict_fn`.

In [ ]:
# Pick a positive (binder) test record to explain.
row = test_df[test_df['binder'] == 1].iloc[0]
peptide = row['peptide']
print('explaining peptide:', peptide, '(binder)')

def predict_fn(frame):
    return model.predict_proba(frame)

attr = residue_attribution(predict_fn, row, field='peptide', cfg=cfg)
attr = np.asarray(attr, dtype=float)
print('attribution length:', len(attr), '== len(peptide):', len(peptide))
print('per-residue importance:', np.round(attr, 3))

## Map importance onto MHC-I anchors P2 and PΩ

For HLA class I, peptide positions **2** and the **C-terminus (PΩ)** are the
dominant anchor residues. `anchor_importance` reports whether the model's attention
concentrates there.

In [ ]:
positions = map_to_positions(attr, peptide)
print('pos  residue  importance')
for pos, residue, importance in positions:
    tag = ''
    if pos == 2:
        tag = '  <- P2 anchor'
    elif pos == len(peptide):
        tag = '  <- POmega anchor'
    print(f'{pos:3d}    {residue}        {importance:7.3f}{tag}')

anchors = anchor_importance(attr, peptide)
print()
print('anchor summary:', anchors)

## Plot per-residue importance

Guarded so it never hard-fails if plotting is unavailable. Uses the Agg backend and
saves a PNG (never `plt.show()`).

In [ ]:
try:
    from tcr_cliff.interpret import plot_residue_importance
    out = plot_residue_importance(attr, peptide, 'peptide_importance.png')
    print('saved', out)
except Exception as exc:  # pragma: no cover - plotting optional
    print('plot skipped:', exc)

**Takeaway.** When the model is right for the right reasons, importance concentrates
on P2/PΩ and on the CDR3b contact residues - and crucially on the single position
that defines each cliff (notebook 03).

### Next
Continue to **07_usecase_bispecific_crossreactivity** for a worked screen.